[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MLManuals/Deep-Learning/blob/main/chapter02_the_artificial_neuron/notebook.ipynb)

# Chapter 2: The Artificial Neuron

Run the cell below once, first, to install any missing packages (only needed on Google Colab; skip it if you're running locally and already have them).

In [ ]:
%pip install -q torch numpy scikit-learn

On a summer day in July 1958, the United States Navy invited reporters in Washington to watch a computer learn. The computer was an IBM 704, a machine that weighed about five tons and filled a whole room, and the demonstration looked almost too simple to be news. It was fed a series of punched cards, each one marked with a square either on its left side or on its right side, and nobody told it which was which. After about fifty tries it had taught itself to tell the left cards from the right ones.

The man running the demonstration was Frank Rosenblatt, a young psychologist from the Cornell Aeronautical Laboratory in Buffalo, New York, and he called his idea the **Perceptron**. Rosenblatt was not trying to write a cleverer program. He was trying to copy, in the simplest possible way, what a single cell in the brain does, and then let many of those copied cells adjust themselves through experience. The next morning the New York Times ran the headline "New Navy Device Learns by Doing", and its report said the Navy expected the machine to one day walk, talk, see, write, reproduce itself and be conscious of its own existence.

None of that came true for a very long time, and the gap between the promise and the machine would come back to hurt the whole field, as you will see in Chapter 3. Yet the small idea at the heart of the Perceptron turned out to be exactly right. Every one of the 60 million adjustable numbers in AlexNet, which you met in Chapter 1, belongs to a unit that does almost precisely what Rosenblatt's copied brain cell did in 1958.

This chapter opens that unit up. You will see what goes into an artificial neuron, what happens inside it and what comes out, and by the end you will have built one yourself in a few lines of numpy. Before you read on, try to guess the answers to these four questions.

1. **What does a real brain cell do that a computer could copy with simple arithmetic?**
2. **If a neuron listens to several inputs, how does it decide which ones matter more?**
3. **Why might a neuron need a starting number that has nothing to do with its inputs?**
4. **When a single neuron sorts examples into two groups, what shape is the border between them?**

## Inside a Single Neuron

Your brain holds tens of billions of nerve cells, called neurons, and each one does a surprisingly modest job. A neuron has many thin branches, called dendrites, that receive small electrical signals from other neurons. Those signals flow into the cell body, where they add up, and if the total becomes strong enough, the neuron fires a signal of its own along a long cable called the axon, which carries it on to the next neurons in line. If the total is too weak, the neuron stays quiet.

You already know a device that works in exactly this way, which is a smoke alarm. A little smoke drifting past it does nothing at all, and it stays silent while the smoke slowly builds up. Once the amount of smoke crosses a certain level, the alarm goes off. The alarm does not care where each wisp of smoke came from, since it only cares whether the total has crossed its threshold.

In 1943, two researchers named Warren McCulloch and Walter Pitts noticed that this behaviour could be written down as simple arithmetic, and Rosenblatt's Perceptron built directly on their idea. Figure 2.1 puts a real brain cell next to its artificial copy, and the colours show which part of one matches which part of the other.

> 🖼️ **Figure 2.1:** A brain cell and its artificial copy. Signals come in, they are added up, and a decision goes out, and the colours show which part matches which.

**An artificial neuron does three things in order: it receives a few numbers, it adds them up with different amounts of importance, and it decides how strongly to respond.** The rest of this chapter takes those three steps one at a time. One honest warning belongs here before you go further. A real brain cell is far more complicated than this copy, and modern neural networks are not good models of the brain. The brain was the inspiration, but from here on you can think of a neuron simply as a small piece of arithmetic.

## Weights, Bias and the Weighted Sum

To make the arithmetic concrete, imagine a teacher who wants a quick way to guess whether a student is ready for an exam, using just two numbers about the week before it, the hours the student spent studying and the average hours they slept each night. A neuron for this job receives those two numbers as its **inputs**.

The neuron does not treat its inputs equally. It gives each one a **weight**, a number that says how much that input should count. Suppose this teacher believes study matters a little more than sleep, so the weight on study is 0.6 and the weight on sleep is 0.4. The neuron multiplies each input by its weight and adds the results together. A weight can also be negative, which is how a neuron learns that something counts against a decision. If the teacher added a third input for the hours spent on a phone each day, its weight would most likely be below zero.

There is one more number, called the **bias**, which is added at the end no matter what the inputs are. The bias sets how easy the neuron is to convince, a little like the difference between a strict examiner and a generous one. Here the bias is $-5$, which means the study and the sleep together have to be worth more than 5 before the neuron leans towards saying yes. The whole calculation is called the **weighted sum**, usually written with the letter $z$.
$$z = 0.6 \times \text{study} + 0.4 \times \text{sleep} + (-5)$$

Figure 2.2 follows one student, who studied for 6 hours and slept 7 hours a night, through the neuron, and you can check every number in it by hand.

> 🖼️ **Figure 2.2:** One student passing through the neuron. Each input is multiplied by its weight, the bias is added, and the total of 1.4 is above zero, so the neuron says the student is ready.

The same calculation is easy to write in Python. The first cell stores the student's two numbers, the two weights and the bias, and the second cell computes the weighted sum. The function `np.dot` multiplies two lists of numbers position by position and adds up the products, which is exactly the job of a weighted sum.

In [ ]:
import numpy as np

student = np.array([6, 7])        # hours studied, hours slept
weights = np.array([0.6, 0.4])
bias = -5.0

In [ ]:
z = np.dot(weights, student) + bias
print(round(z, 2))

Table 2.1 works through three different students in the same way. Notice how the third student comes very close to the line, because a little more study would have tipped the sum above zero.

**Table 2.1: The weighted sum for three students, with a bias of $-5$ added to each one.**

| Student | Studied, slept | Weighted parts | Weighted sum $z$ | Above zero? |
|---|---|---|---|---|
| A | 6 and 7 | $3.6 + 2.8$ | $1.4$ | Yes |
| B | 2 and 8 | $1.2 + 3.2$ | $-0.6$ | No |
| C | 4 and 6 | $2.4 + 2.4$ | $-0.2$ | No, but close |

## The Activation Step

The weighted sum is just a number, and it can be any size at all. The last step of the neuron, called the **activation**, turns that number into the neuron's answer. Rosenblatt's Perceptron used the simplest activation possible, which is called a **step function**. It works like an ordinary light switch, since anything above zero turns the light fully on and gives an output of 1, and anything at zero or below leaves it off and gives an output of 0.

In [ ]:
def step(z):
    return 1 if z > 0 else 0

print(step(1.4), step(-0.6), step(-0.2))

Student A is ready and students B and C are not. The step function has a problem, though, and you can see it in student C. That student fell short by only 0.2, while student B fell short by 0.6, and yet the step function gives both of them exactly the same answer. It throws away the information about how close a decision was.

A gentler choice is the **sigmoid** function, which works more like a dimmer switch than a light switch. Instead of jumping from 0 to 1, it slides smoothly between them, so a sum far below zero gives an answer near 0, a sum far above zero gives an answer near 1, and a sum of exactly zero gives 0.5, the point of complete uncertainty. You can read the sigmoid's answer as how confident the neuron is.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

for z in [1.4, -0.6, -0.2]:
    print(round(sigmoid(z), 3))

Now the answers carry the closeness that the step function threw away. Student C, at 0.45, sits just below the halfway point, while student B, at 0.354, is noticeably further from it. Figure 2.3 draws both activations and marks where the three students land on each one.

> 🖼️ **Figure 2.3:** The two activations side by side. The step function gives students B and C the same answer, while the sigmoid keeps the difference between them.

That difference matters far more than it seems. In Chapter 6 a network learns by making many tiny changes to its weights and checking whether each change helped. A small change to a weight barely moves student C's weighted sum, and with the step function the output usually does not move at all, so the network gets no hint that it is getting warmer. The sigmoid always moves a little, and that small movement is the signal that learning feeds on. **A smooth activation lets every small change in a weight show up as a small change in the answer, and that is what makes learning possible.** Chapter 4 explores activations properly, including the one that most modern networks use instead of either of these.

## A Neuron in a Few Lines of numpy

You now have every piece of a neuron, and joining them takes only a few lines. The function below takes an example, a set of weights and a bias, computes the weighted sum, and passes it through the step activation. The step function is used here so that the neuron gives a clear yes or no, the way the teacher wants.

In [ ]:
def neuron(x, weights, bias):
    z = np.dot(weights, x) + bias
    return step(z)

print(neuron(np.array([6, 7]), weights, bias))
print(neuron(np.array([2, 8]), weights, bias))

A teacher has a whole class, not one student, so the next cell builds a class of 60 students from a fixed seed. Each student gets random hours of study between 0 and 10 and random hours of sleep between 3 and 10. In real life, whether a student is ready depends on those same two habits plus some luck on the day, so the cell adds a little random noise before deciding who really passed.

In [ ]:
rng = np.random.default_rng(3)
study = rng.uniform(0, 10, 60).round(1)
sleep = rng.uniform(3, 10, 60).round(1)
X = np.column_stack([study, sleep])
noise = rng.normal(0, 0.6, 60)
passed = ((0.6 * study + 0.4 * sleep - 5 + noise) > 0).astype(int)
print(X[:3])
print(passed[:3])

Each row of `X` is one student. You could loop over all 60 rows and call `neuron` on each one, but numpy has a faster way. The `@` symbol multiplies a whole table by the weights in one go, computing the weighted sum of every row at once, and comparing the result with zero then gives the step function's answer for the whole class.

In [ ]:
z = X @ weights + bias
guess = (z > 0).astype(int)
print(guess[:3])
print((guess == passed).mean().round(2))

The neuron is right about 95 percent of the class, and its second guess shows one of the few it gets wrong, a student who studied for only 2.4 hours but slept well, had a good day and passed anyway. Figure 2.4 shows the whole class and reveals something important about every neuron. All the students on one side of a single straight line get a yes, and all the students on the other side get a no. That line is where the weighted sum equals exactly zero, and it is called the **decision boundary**.

> 🖼️ **Figure 2.4:** The class of 60 students and the neuron's decision boundary. Everything on one side of the straight line gets a yes, and the three mistakes all sit close to the line.

The three mistakes all sit close to the line, which is where a little luck on the day was enough to change the result. More importantly, the border between yes and no is a straight line, and it would be a straight line for any weights and any bias you chose. Changing the weights tilts the line and changing the bias slides it, but nothing can bend it. **A single neuron with a step activation always splits its examples with one straight line, so it can only answer questions whose two groups a straight line can separate.** Chapter 3 shows a famous question where no straight line can ever work.

You may also have noticed that you chose the weights and the bias in this chapter yourself, the way the teacher chose them from experience. Rosenblatt's machine found its own, by nudging its weights a little every time it made a mistake, and finding good weights automatically is what Chapters 5, 6 and 7 are about.

## Practice and Review

Reading about deep learning is only half of learning it. Pulling an idea back out of your memory makes it stick much better than reading it again, so this section gives you two kinds of practice, one for your hands and one for your memory.

### Try It Yourself

1. **Make the teacher stricter.** Change the bias from $-5$ to $-6$, run the class cells again, and count how many students are now called ready. Work out which way the boundary in Figure 2.4 moved.
2. **Add a phone input.** Give each student a third input, the daily hours on a phone from `rng.uniform(0, 5, 60)`, with a weight of $-0.3$, and check whether student A, with 3 hours of phone use, is still ready.
3. **Swap the activation.** Pass the class's weighted sums through `sigmoid` instead of the step function, print the five students closest to 0.5, and find them in Figure 2.4.

### Check Your Understanding

Try to answer each question from memory before you look at the answers that follow.

1. **What are the three steps every artificial neuron performs?**
2. **What does a negative weight tell you about an input?**
3. **What job does the bias do, in everyday words?**
4. **Why does the step function make learning hard, and how does the sigmoid help?**
5. **What shape is a single neuron's decision boundary, and what can change it?**

### Answers

1. It receives a few numbers, adds them up with a weight on each and a bias, and passes the total through an activation.
2. That the input counts against the decision, so a larger input pulls the weighted sum further down.
3. It sets how easy the neuron is to convince, like a strict or a generous examiner.
4. The step function gives the same answer whether a decision was close or not, so a tiny change in a weight usually changes nothing, while the sigmoid shows every small change.
5. It is always a straight line, which the weights can tilt and the bias can slide, but which nothing can bend.

## What Comes Next

You now know what a neuron is made of, and why a single neuron can only ever draw one straight line.

- **Chapter 3, From Neurons to a Network,** joins neurons into layers and meets a puzzle no straight line can solve.
- **Chapter 4, Activation Functions and Nonlinearity,** shows why the bend inside each neuron decides whether a network can learn.

Chapter 3 begins in 1969, with a short book from MIT that came close to ending neural network research.